# Rational design and structural characterization of bioactive molecules - UniMi 2025/2026

In this tutorial we will present you Boltz2 and its use, as well as guidelines on how to critically evaluate the quality of the generated predicted structures.

All data and information are available from this tutorial [GitHub page](https://github.com/LucaChiesa/UniMi_2024_AlphaFold_Tutorial).

##  Environment setup

In [ ]:
# @title Install dependencies { display-mode: "form" }
import os
print("Installing Boltz2")
if not os.path.isfile("BOLTZ_READY"):
    os.system("pip install boltz[cuda] -U")
    os.system("pip install py3Dmol")
    os.system("touch BOLTZ_READY")

if not os.path.isfile('DATA_READY'):
    os.system("git clone -q https://github.com/LucaChiesa/UniMi_2024_AlphaFold_Tutorial")
    os.system("touch DATA_READY")
print("Tutorial material downloaded")

import torch
if torch.cuda.is_available():
    print('Running on GPU')
else:
    print("WARNING: no GPU detected, will be using CPU")

In [ ]:
# @title Set local variables { display-mode: "form" }
github = 'UniMi_2024_AlphaFold_Tutorial/'
protein_structures = f'{github}protein_structures'
reference_structures = f'{github}reference_structures'
inputs = f'{github}inputs'

import warnings
warnings.filterwarnings('ignore')
from google.colab import files
from UniMi_2024_AlphaFold_Tutorial.utils import *

In [ ]:
# @title Javascript test { display-mode: "form" }
import py3Dmol
test_file = f"{protein_structures}/1ezg.pdb"
view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width = 400, height = 300)
view.addModel(open(test_file,'r').read(),'pdb')
view.setStyle({'model':0},{'cartoon': {'color':'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
# @title  Display the FASTA file  { display-mode: "form" }
import textwrap
with open(f"{inputs}/example_proteins.fasta") as fasta_inp:
    fasta_strs = fasta_inp.read().split()
for fasta_str in fasta_strs:
    print('\n'.join(textwrap.wrap(fasta_str)))

## Example system - Segment of a viral protein
Precalculated example of the results obtained using Boltz2

In [ ]:
# @title  Display the MSA  { display-mode: "form" }
#@markdown Display the first 20 sequences which form the MSA
import pandas as pd
import matplotlib
msa = pd.read_csv(f'{protein_structures}/boltz_results_example_protein/msa/example_protein_0.csv')['sequence']
for seq in msa:
    print(seq)

In [ ]:
# @title  Display MSA coverage  { display-mode: "form" }
%matplotlib inline
msa = np.array([[s for s in seqs[:len(seq)]] for seqs in msa.to_list()])
plot_msa_v2(msa)
plt.show()
plt.close()

In [ ]:
# @title Display 3D structure { display-mode: "form" }
#from litaf.utils import show_pdb
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
tag = rank_num - 1

#@markdown Click on any atom to show a label with the residue name, the residue number, and the plDDT value.<br/>Click a second time on the same atom to hide the label.


vprot_path = "boltz_results_example_protein/predictions/example_protein/"

pdb_filename = f"{protein_structures}/{vprot_path}/example_protein_model_0.cif"

show_pdb(pdb_filename, 1, show_sidechains, show_mainchains, color, extension='cif').show()
if color == "lDDT":
    plot_plddt_legend().show()

In [ ]:
# @title Plot plDDT & PAE { display-mode: "form" }

plddt = np.load(f"{protein_structures}/{vprot_path}/plddt_example_protein_model_0.npz")
pae = np.load(f"{protein_structures}/{vprot_path}/plddt_example_protein_model_0.npz")

plddts_plot = plot_plddts([plddt['plddt']*100])
pae_plot = plot_paes([pae['pae']])

plddts_plot.savefig(f"{protein_structures}/{vprot_path}/plddts_plot.png")
pae_plot.savefig(f"{protein_structures}/{vprot_path}/pae_plot.png")

## SARS-CoV-2 3C-like protease
We run full calculations for this one, from using MMseqs2 to evaluating the complex quality

In [ ]:
# @title Input file { display-mode: "form" }

from pathlib import Path
import logging
import os

sars_cov_path = 'boltz_results_sars-cov2_complex/predictions/sars-cov2_complex'

with open(f'{inputs}/sars-cov2_complex.yaml') as f:
    input_cont = f.read()
print(input_cont)

In [ ]:
# @title Run predictions { display-mode: "form" }
os.system(f"boltz predict {inputs}/sars-cov2_complex.yaml --use_msa_server")

In [ ]:
# @title Display MSA coverage { display-mode: "form" }
%matplotlib inline
msa = pd.read_csv(f'{protein_structures}/boltz_results_sars-cov2_complex/msa/sars-cov2_complex_0.csv')['sequence']
msa = np.array([[s for s in seqs[:len(seq)]] for seqs in msa.to_list()])
plot_msa_v2(sarscov_monomer.feature_dict)
plt.show()
plt.close()

In [ ]:
# @title Display 3D structure { display-mode: "form" }
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

pdb_filename = f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0.cif"

show_pdb(pdb_filename, 1, show_sidechains, show_mainchains, color, extension='cif').show()
if color == "lDDT":
    plot_plddt_legend().show()

In [ ]:
# @title Plot plDDT & PAE { display-mode: "form" }
plddt = np.load(f"{protein_structures}/{sars_cov_path}/plddt_sars-cov2_complex_model_0.npz")
pae = np.load(f"{protein_structures}/{sars_cov_path}/pae_sars-cov2_complex_model_0.npz")

plddts_plot = plot_plddts([plddt['plddt']*100])
pae_plot = plot_paes([pae['pae']])

plddts_plot.savefig(f"{protein_structures}/{sars_cov_path}/plddts_plot.png")
pae_plot.savefig(f"{protein_structures}/{sars_cov_path}/pae_plot.png")

In [ ]:
# @title Align AlphaFold models on reference structures { display-mode: "form" }
ref_b_file = f'{reference_structures}/6XQS.pdb'
ref_b_aligned = f'{reference_structures}/6XQS_aligned.pdb'
ref_a_file = f'{reference_structures}/6WQF.pdb'
ref_a_aligned = f'{reference_structures}/6WQF_aligned.pdb'

print("Align references")
align_on_ref(ref_a_file, ref_b_file, 'SV6')
align_on_ref(f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0.cif", ref_b_file, 'SV6', example = True)

In [ ]:
# @title Comparison between the free protein (PDBID: 6WQF) and the ligand bound protein (PDBID: 6XQS)  { display-mode: "form" }

holo_cartoon_color = "green" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
apo_cartoon_color = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
holo_sidechain_color = "cyan" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
apo_sidechain_color = "yellow" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

#@markdown Click on any atom to show a label with the residue name, the residue number, and the atom name.<br/>Click a second time on the same atom to hide the label.

show_aligned(ref_a_aligned, ref_b_aligned, 'SV6',
             model_cartoon_color = apo_cartoon_color,
             ref_cartoon_color = holo_cartoon_color,
             ligand_color = ligand_color+'Carbon',
             model_sidecahin_color = apo_sidechain_color+'Carbon',
             ref_sidecahin_color = holo_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(name_model = 'Apo', name_ref = 'Holo',
             model_cartoon_color = apo_cartoon_color,
             ref_cartoon_color = holo_cartoon_color,
             ligand_color = ligand_color,
             model_sidecahin_color = apo_sidechain_color,
             ref_sidecahin_color = holo_sidechain_color).show()

In [ ]:
# @title Comparison between the AlphaFold model and the free protein (PDBID: 6WQF) { display-mode: "form" }
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}

reference_cartoon_color = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_cartoon_color = "magenta" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
reference_sidechain_color = "yellow" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_sidechain_color = "purple" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

aligned_model = f"{output_path}/ranked_{rank_num - 1}_aligned.pdb"

show_aligned(aligned_model, ref_a_aligned, 'GST',
             model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color+'Carbon',
             model_sidecahin_color = AF_model_sidechain_color+'Carbon',
             ref_sidecahin_color = reference_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color,
             model_sidecahin_color = AF_model_sidechain_color,
             ref_sidecahin_color = reference_sidechain_color).show()

In [ ]:
# @title Comparison between the AlphaFold model and the ligand bound protein (PDBID: 6XQS)  { display-mode: "form" }
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}

reference_cartoon_color = "green" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_cartoon_color = "magenta" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
reference_sidechain_color = "cyan" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_sidechain_color = "purple" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

aligned_model = f"{output_path}/ranked_{rank_num - 1}_aligned.pdb"
show_aligned(aligned_model, ref_b_aligned, 'SV6',
             model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color+'Carbon',
             model_sidecahin_color = AF_model_sidechain_color+'Carbon',
             ref_sidecahin_color = reference_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color,
             model_sidecahin_color = AF_model_sidechain_color,
             ref_sidecahin_color = reference_sidechain_color).show()

In [ ]:
# @title Package and Download results { display-mode: "form" }
results_zip = f"3CL-proteinase_result.zip"
os.system(f"zip -r {results_zip} boltz_results_sars-cov2_complex")
files.download(results_zip)